# Phase 2: stage the invest_central population

Freezes the D2 population (`InvestCustomer__pc = 'True' AND InvestmentStatus__pc set AND <> 'Owner'`)
into `crm_imp_person_accounts`, one row per ContactPointEmail, CPE Id prefilled via
`PartyID__c = PersonContactId`. Two batch ids because the consent script writes one
constant `PrivacyConsentStatus` per run.

**This notebook only writes to the local MySQL staging table. It never touches Salesforce.**

Prerequisites: mirrors refreshed (`camping-grubhof-import/refresh_sf_mirrors.py`),
then `01_create_mirror_indexes.sql` re-applied (the refresh drops the indexes).

Staged 2026-08-25: **6,212 OptIn / 10 OptOut** (fresh mirror; the 2026-08-19 audit
predicted 6,146 accounts and 11 exclusion matches — population grew by 3 accounts
during the week and one exclusion match no longer holds; investigated, cause of the
11→10 not established, parked by Arsal 2026-08-25).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root: config, mysql_client

from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_OPTIN = "2026-08-25_invest_central_optin"
BATCH_OPTOUT = "2026-08-25_invest_central_optout"
EXCLUSION_XLS = Path("data/20260820_FMTG Invest_exclusion list Salesforce.xls")

db = MySQLClient(load_mysql_config())
print("connected")

## 1. Mirror freshness

`MAX(LastModifiedDate)` should be within a day of the refresh; if it is stale, stop and re-run
the refresh before staging.

In [ ]:
for table in ("crm_person_account_sfid_prod", "crm_cp_email_sfid_prod", "crm_consent_sfid_prod"):
    row = db.fetch_one(f"SELECT COUNT(*) AS n, MAX(LastModifiedDate) AS newest FROM {table}")
    print(f"{table:35s} {row['n']:>12,}  newest: {row['newest']}")

## 2. Population counts

Expected from the 2026-08-19 audit: 6,146 accounts; CPE cardinality 0→1 / 1→6,071 / 2→74;
6,219 CPE rows. Small drift after a refresh is possible — investigate anything larger than a handful.

In [ ]:
POPULATION_WHERE = """
    a.InvestCustomer__pc = 'True'
    AND a.InvestmentStatus__pc IS NOT NULL
    AND a.InvestmentStatus__pc <> ''
    AND a.InvestmentStatus__pc <> 'Owner'
"""

cardinality = db.fetch_df(f"""
    SELECT t.n_cpe, COUNT(*) AS n_accounts
    FROM (
        SELECT a.Id, COUNT(e.Id) AS n_cpe
        FROM crm_person_account_sfid_prod a
        LEFT JOIN crm_cp_email_sfid_prod e ON e.PartyID__c = a.PersonContactId
        WHERE {POPULATION_WHERE}
        GROUP BY a.Id
    ) t
    GROUP BY t.n_cpe ORDER BY t.n_cpe
""")
print(cardinality.to_string(index=False))

expected_rows = int((cardinality["n_cpe"] * cardinality["n_accounts"]).sum())
expected_accounts = int(cardinality["n_accounts"].sum())
print(f"\npopulation accounts: {expected_accounts:,}  |  consent rows to stage: {expected_rows:,}")

## 3. Stage the OptIn batch

One row per (account, CPE). The account with zero CPEs produces no row (D7: excluded by
construction). Guarded: refuses to run if either batch id already has rows — rerun after a
mistake means deleting the batch first, deliberately, not re-executing the cell.

In [ ]:
existing = db.fetch_one(
    "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id IN (%s, %s)",
    (BATCH_OPTIN, BATCH_OPTOUT),
)["n"]
assert existing == 0, f"{existing} rows already staged under these batch ids — not re-inserting"

inserted = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id, sf_cp_email_id,
         consent_invest, investment_status)
    SELECT
        'update', %s, 0, 'FMTG Invest',
        a.LastName, e.EmailAddress,
        a.Id, a.PersonContactId, e.Id,
        1, a.InvestmentStatus__pc
    FROM crm_person_account_sfid_prod a
    JOIN crm_cp_email_sfid_prod e ON e.PartyID__c = a.PersonContactId
    WHERE {POPULATION_WHERE}
""", (BATCH_OPTIN,))

print(f"staged {inserted:,} rows under {BATCH_OPTIN}")
assert inserted == expected_rows, f"staged {inserted} but section 2 predicted {expected_rows}"

## 4. Exclusion list → OptOut batch

The delivered list (`data/`, PII, gitignored, password-protected) is decrypted in memory.
Matching is trim+lowercase on email, the same rule the audit used.

The move is **per account, not per email** (review finding 2026-08-25): an excluded
person with a second CPE under a different address must have BOTH staged rows moved
to OptOut, otherwise the second row would write an OptIn for someone on the exclusion
list. The audit predicted 11 matched accounts; the fresh mirror yields 10 (parked).

In [ ]:
import getpass
import io

import msoffcrypto
import pandas as pd

buf = io.BytesIO()
with open(EXCLUSION_XLS, "rb") as f:
    of = msoffcrypto.OfficeFile(f)
    of.load_key(password=getpass.getpass("exclusion list password: "))
    of.decrypt(buf)
buf.seek(0)

excl = pd.read_excel(buf)
excl_emails = sorted({e.strip().lower() for e in excl["email"].dropna()})
assert excl_emails, "exclusion list decrypted to zero emails — wrong sheet or column?"
print(f"exclusion list: {len(excl)} rows, {len(excl_emails)} unique emails")

In [ ]:
placeholders = ",".join(["%s"] * len(excl_emails))

# Alle Zeilen der getroffenen ACCOUNTS bewegen, nicht nur die getroffene
# E-Mail-Zeile: sonst bleibt die Zweit-CPE eines Ausgeschlossenen im OptIn.
matched_accounts = [r["sf_account_id"] for r in db.fetch_all(f"""
    SELECT DISTINCT sf_account_id
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s AND LOWER(TRIM(email)) IN ({placeholders})
""", (BATCH_OPTIN, *excl_emails))]
print(f"exclusion emails match {len(matched_accounts)} staged accounts")

if matched_accounts:
    acc_ph = ",".join(["%s"] * len(matched_accounts))
    moved = db.execute(f"""
        UPDATE crm_imp_person_accounts
        SET _batch_id = %s
        WHERE _batch_id = %s AND sf_account_id IN ({acc_ph})
    """, (BATCH_OPTOUT, BATCH_OPTIN, *matched_accounts))
    print(f"moved {moved} rows to {BATCH_OPTOUT}")
    assert moved >= len(matched_accounts)

assert 8 <= len(matched_accounts) <= 13, (
    f"{len(matched_accounts)} matched accounts is far off the audit's 11 — "
    "email matching is probably broken, do not proceed"
)

## 5. Verification and predicted after-state

These numbers are the phase 6 contract: after the load, the live consent pivot must show
exactly `staged OptIn rows` invest_central OptIn consents and `staged OptOut rows` OptOut.

In [ ]:
summary = db.fetch_df("""
    SELECT _batch_id,
           COUNT(*) AS rows_staged,
           COUNT(DISTINCT sf_account_id) AS accounts,
           COUNT(DISTINCT sf_cp_email_id) AS cpes,
           SUM(sf_cp_email_id IS NULL) AS null_cpe_rows
    FROM crm_imp_person_accounts
    WHERE _batch_id IN (%s, %s)
    GROUP BY _batch_id
""", (BATCH_OPTIN, BATCH_OPTOUT))
print(summary.to_string(index=False))

dupes = db.fetch_one("""
    SELECT COUNT(*) AS n FROM (
        SELECT sf_cp_email_id
        FROM crm_imp_person_accounts
        WHERE _batch_id IN (%s, %s)
        GROUP BY sf_cp_email_id HAVING COUNT(*) > 1
    ) d
""", (BATCH_OPTIN, BATCH_OPTOUT))["n"]
assert dupes == 0, f"{dupes} duplicate sf_cp_email_id values staged"
assert int(summary["null_cpe_rows"].astype(int).sum()) == 0

# Kein Konto darf Zeilen in BEIDEN Batches haben (per-Account-Kontrakt).
split = db.fetch_one("""
    SELECT COUNT(*) AS n FROM (
        SELECT sf_account_id FROM crm_imp_person_accounts
        WHERE _batch_id IN (%s, %s)
        GROUP BY sf_account_id HAVING COUNT(DISTINCT _batch_id) > 1
    ) s
""", (BATCH_OPTIN, BATCH_OPTOUT))["n"]
assert split == 0, f"{split} accounts have rows in both batches"
print("\nno duplicate CPEs, no NULL CPE ids, no account split across batches — staging frozen")

## Next

- Phase 3: `insert_consents_invest.py` (Bulk API loader, CLI script — dry-run against these batches).
- Phase 4: one probe record in prod, business sign-off (see `run_invest_consents.ipynb`).
- Phase 5: bulk load. **Does not run without Arsal's explicit go-ahead.**